# Graphene/Ni(111) Interface: Registry, Separation and Work of Adhesion

## 0. Introduction

This notebook reproduces the structure and energetics of graphene on Ni(111) following the review:

> **Arjun Dahal, Matthias Batzill**
> "Graphene–nickel interfaces: a review"
> Nanoscale, 6(5), 2548. (2014)
> [DOI: 10.1039/c3nr05279f](https://doi.org/10.1039/c3nr05279f)

The review's structural facts (its section 2.1): graphene locks into a 1×1 registry on Ni(111);
LEED I–V and ion scattering identify the adsorbed structure as one carbon **atop** a first-layer Ni
and the other in the **fcc hollow**, 0.211 nm above the surface with a 0.005 nm buckling in which
the atop carbon sits further out. Its computed numbers come from
[Lahiri et al., New J. Phys. 13, 025001 (2011)](https://doi.org/10.1088/1367-2630/13/2/025001)
(open access), whose Table 1 is the quantitative target here:

| interface | work of adhesion (J/m²) | separation (Å) |
|---|---|---|
| fcc (atop + fcc hollow) | 0.81 | 2.16 |
| hcp (atop + hcp hollow) | 0.77 | 2.17 |
| hollow (fcc + hcp hollows) | 0.31 | 3.26 |

The four candidate registries, in the review's own Fig. 1:

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/optimization/optimization_interface_film_xy_position_graphene_nickel/0-figure-from-manuscript.webp" alt="The four registries of graphene on a close-packed metal surface" width="600"/>

The bridge registry (d) is not quantified in either paper — it is included here as an extra point
beyond the published set.

The published calculation (Lahiri et al., section 2.2) used **LDA, spin-polarized, with geometry
relaxation** — five Ni layers with the bottom two fixed — because "GGA does not provide an adequate
description of Ni–graphene bonding". This notebook follows that recipe in two tiers:

- **Fast (here, in minutes):** each registry relaxed with the
  [MACE-MP](https://github.com/ACEsuit/mace) machine-learned force field (+D3), with the bottom
  substrate layers fixed as in the paper, heights only; same-cell references give the work of
  adhesion. MACE is PBE-trained and misses the paper's numbers on this interface: chemisorption
  several times too weak, the separation short, the atop carbon buckled the wrong way. What it
  delivers in minutes is the registry set, the two-branch (chemisorbed / dispersion-bound) energy
  landscape and the starting geometries for the precise tier; its table prints beside the paper's
  so the gap shows.
- **Precise (platform jobs):** the paper's functional — **LDA** (pz, ultrasoft), spin-polarized,
  **fixed-cell relaxation**, no dispersion correction — for each registry plus the two same-cell
  references the work of adhesion needs; the relaxed geometry is read back and compared too.

**Prerequisite:** run
[optimization_interface_film_xy_position_graphene_nickel.ipynb](optimization_interface_film_xy_position_graphene_nickel.ipynb)
first — it creates and saves the base interface material this notebook loads.

## 1. Prepare the Environment
### 1.1. Install Packages


In [ ]:
from mat3ra.notebooks_utils.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Set Parameters


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "./uploads"
BASE_MATERIAL_NAME = "Graphene_Nickel_interface"  # created by the companion structure notebook

# 4. MLFF parameters
MACE_MODEL_FAMILY = "MACE-MP-0"
MACE_MODEL = "large"
MACE_DISPERSION = True
MACE_DEFAULT_DTYPE = "float64"
MACE_DEVICE = "cpu"

# 5. Separation scan, in Angstrom — brackets both published minima (2.16 and 3.26 A)
Z_SCAN_START = 1.8
Z_SCAN_STOP = 4.3
Z_SCAN_STEP = 0.25
CHEMISORBED_BELOW = 2.6  # boundary between the chemisorbed and dispersion-bound branches

# 6. Relaxation — the paper's scheme; the buckling is one of the published numbers
FMAX = 0.02  # eV/A
FROZEN_SUBSTRATE_LAYERS = 2

# 7. Workflow parameters
WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Fixed-cell Relaxation (Gr/Ni registry)"

# Method parameters — the published setup (Lahiri et al., section 2.2) where the platform can
# express it: LDA, spin-polarized, relaxed, no dispersion correction.
PSEUDOPOTENTIAL_TYPE = "us"
FUNCTIONAL = "pz"
MODEL_SUBTYPE = "lda"
ECUTWFC = 40   # GBRV's published pair
ECUTRHO = 200
SCF_KGRID = [12, 12, 1]  # multiple of 3 keeps K on the mesh; dense for a metal
STARTING_MAGNETIZATION = {"Ni": 0.7}  # near the bulk moment

# SCF settings for a spin-polarized metal slab
SMEARING = "mv"
DEGAUSS = 0.01  # Ry
ADDITIONAL_PARAMETERS = {
    "electrons": {
        "mixing_mode": "local-TF",
        "mixing_beta": 0.2,
        "electron_maxstep": 200,
    },
}

# 8. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 9. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30


## 2. Load the Base Interface

The base interface is created by the companion structure notebook and saved into `uploads/`.
It is required — this notebook does not substitute another material.


In [ ]:
from mat3ra.made.material import Material
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance
from mat3ra.made.tools.convert import to_ase
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.made.tools.modify import interface_get_part
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

base_interface = load_material_from_folder(FOLDER, BASE_MATERIAL_NAME)
if base_interface is None:
    raise RuntimeError(
        f"'{BASE_MATERIAL_NAME}' not found in {FOLDER} — run "
        "optimization_interface_film_xy_position_graphene_nickel.ipynb first."
    )

film_part = interface_get_part(base_interface, part=InterfacePartsEnum.FILM)
substrate_part = interface_get_part(base_interface, part=InterfacePartsEnum.SUBSTRATE)
film_elements = set(film_part.basis.elements.values)
substrate_elements = set(substrate_part.basis.elements.values)
substrate_indices = [i for i, label in enumerate(base_interface.basis.labels.values)
                     if label == InterfacePartsEnum.SUBSTRATE.value]
n_carbon = len(film_part.basis.elements.values)
measured_gap = get_average_interlayer_distance(
    to_ase(base_interface), InterfacePartsEnum.SUBSTRATE.value, InterfacePartsEnum.FILM.value)

print(f"Material:  {base_interface.name}")
print(f"Atoms:     {len(base_interface.basis.elements.values)} "
      f"({n_carbon} film C, {len(substrate_indices)} substrate Ni)")
print(f"Film-substrate separation as built: {measured_gap:.3f} A")

visualize([{"material": base_interface, "title": base_interface.name}], repetitions=[3, 3, 1], rotation="-90x")


## 3. Place the Film at the High-Symmetry Registries

The registries are defined by where carbon atoms sit relative to the Ni(111) surface sites:
**top** (above a first-layer Ni), **hcp hollow** (above a second-layer Ni), **fcc hollow**
(above a third-layer Ni), and **bridge** — the C–C bond midpoint sits over a first-layer Ni
(Fig. 1d), so neither carbon lands on a named site.
The sites are measured from the structure itself — the top three Ni layers — and the film is
translated so one carbon sublattice lands on each site in turn.


In [ ]:
import numpy as np
from mat3ra.made.tools.analyze.other import get_closest_site_id_from_coordinate_and_element
from mat3ra.made.tools.helpers import SurfaceSiteAnalyzer, get_film_site_occupation
from mat3ra.made.tools.modify import interface_displace_part

surface = SurfaceSiteAnalyzer(material=substrate_part)
film_z = float(np.mean([c[2] for c in film_part.coordinates_array]))  # crystal (fractional) coordinate
anchor = get_closest_site_id_from_coordinate_and_element(film_part, [1 / 3, 2 / 3, film_z], "C")
film_cartesian = film_part.clone()
film_cartesian.to_cartesian()
carbon_xy = [np.array(c[:2]) for c in film_cartesian.coordinates_array]

# The anchor carbon sits on the hollow its registry name doesn't mention: atop_fcc sends it to fcc
# (the other carbon lands atop), atop_hcp sends it atop (the other lands on hcp), hollow sends it
# to hcp (the other lands on fcc).
ANCHOR_SITE = {"atop_fcc": "fcc", "atop_hcp": "atop", "hollow": "hcp"}
displacements = {label: surface.get_displacement_to_site(carbon_xy[anchor], site) for label, site in ANCHOR_SITE.items()}
displacements["bridge"] = surface.get_displacement_to_site(np.mean(carbon_xy, axis=0), "atop")

REGISTRY_SITES = {"atop_fcc": {"atop", "fcc"}, "atop_hcp": {"atop", "hcp"}, "hollow": {"fcc", "hcp"}}
for label, shift in displacements.items():
    occupied = get_film_site_occupation(interface_displace_part(base_interface, displacement=list(shift)), surface)
    print(f"{label:<10} carbons on {sorted(str(site) for site in occupied.values())}   shift (A): {np.round(shift[:2], 3) + 0.0}")
    if label in REGISTRY_SITES:
        assert set(occupied.values()) == REGISTRY_SITES[label], f"{label}: carbons on {sorted(map(str, occupied.values()))}"
    else:
        assert set(occupied.values()) == {None}, f"{label}: carbons on {sorted(map(str, occupied.values()))}"


In [ ]:
def film_at(registry_label, plane_distance):
    displacement = displacements[registry_label] + np.array([0.0, 0.0, plane_distance - measured_gap])
    return interface_displace_part(base_interface, displacement=list(displacement))

preview = []
for label in displacements:
    m = film_at(label, measured_gap)
    m.name = f"{BASE_MATERIAL_NAME} {label}"
    preview.append({"material": m, "title": label})

visualize(preview, repetitions=[2, 2, 1])

## 4. Fast Tier: Relax Each Registry with MACE

Each registry is bracketed by a rigid scan, then **relaxed** — positions move along z only, with
the deepest substrate layers fixed, for the interface and both same-cell references (bare Ni slab,
free-standing graphene) alike. For the paper's symmetric registries this equals full relaxation,
since in-plane forces vanish by symmetry; for the bridge, an in-plane saddle, it is what keeps the
point defined. Relaxing all three under the same constraint turns total energies into a work of
adhesion: W = (E_slab + E_graphene − E_interface) / A. Every atom's xy is held fixed by the
z-only constraint, so no film can slide into a neighbouring registry in this tier; the occupation
is still re-measured because the platform tier below relaxes every coordinate freely, where a
slide is possible, and there a structure that lands in a different registry is dropped rather
than reported under the wrong name. Distances follow the paper's convention: the averaged carbon
height above the averaged top-Ni height; buckling is the height difference between the two
carbons, positive when the atop carbon sits further out.


In [ ]:
import importlib.util

from mat3ra.notebooks_utils.mlff import create_mlff_calculator

dispersion_available = importlib.util.find_spec("torch_dftd") is not None
dispersion_active = MACE_DISPERSION and dispersion_available
if MACE_DISPERSION and not dispersion_available:
    print("torch-dftd is not available here: the fast tier runs WITHOUT dispersion — the")
    print("GGA-level picture the manuscript describes as inadequate for this interface.")

calculator = create_mlff_calculator(
    "mace",
    {
        "family": MACE_MODEL_FAMILY,
        "model": MACE_MODEL,
        "dispersion": dispersion_active,
        "default_dtype": MACE_DEFAULT_DTYPE,
        "device": MACE_DEVICE,
    },
)


In [ ]:
from mat3ra.made.tools.calculate import calculate_adhesion_energy, calculate_total_energy
from mat3ra.made.tools.helpers import get_atom_indices_by_layer
from mat3ra.notebooks_utils.mlff.relaxation import relax_material

EV_PER_A2_TO_J_PER_M2 = 16.0217663
layers = get_atom_indices_by_layer(base_interface)
frozen = [i for layer in layers[:FROZEN_SUBSTRATE_LAYERS] for i in layer if i in substrate_indices]

def relax_registry(material):
    return relax_material(material, calculator, fmax=FMAX, fixed_atom_indices=frozen, along_z_only=True)

def carbon_sites_and_buckling(interface):
    """The sites the carbons occupy after relaxation, and the atop carbon's height above the other
    (None when no carbon is atop — then there is no sign to report)."""
    occupied = get_film_site_occupation(interface, surface)
    cartesian = interface.clone()
    cartesian.to_cartesian()
    heights = {i: cartesian.coordinates_array[i][2] for i in occupied}
    atop = [i for i, site in occupied.items() if site == "atop"]
    buckling = None if not atop else float(heights[atop[0]] - next(z for i, z in heights.items() if i != atop[0]))
    return set(occupied.values()), buckling

def buckling_text(buckling):
    return "   —   " if buckling is None else f"{buckling:+.3f}"

substrate_layers = get_atom_indices_by_layer(substrate_part)
slab_relaxed = relax_material(substrate_part, calculator, fmax=FMAX,
                              fixed_atom_indices=[i for layer in substrate_layers[:FROZEN_SUBSTRATE_LAYERS] for i in layer],
                              along_z_only=True)
film_relaxed = relax_material(film_part, calculator, fmax=FMAX, along_z_only=True)


In [ ]:
distances = np.arange(Z_SCAN_START, Z_SCAN_STOP + 1e-9, Z_SCAN_STEP)

scan_results = {}
for label in displacements:
    energies = np.array([calculate_total_energy(film_at(label, float(d)), calculator) for d in distances])

    # a bracketed minimum: the lowest scanned point of a branch that is not a window edge
    starts = {}
    for branch, in_branch in (("chem", distances < CHEMISORBED_BELOW), ("phys", distances >= CHEMISORBED_BELOW)):
        i = int(np.where(in_branch)[0][np.argmin(energies[in_branch])])
        if 0 < i < len(distances) - 1 and energies[i] <= min(energies[i - 1], energies[i + 1]):
            starts[branch] = float(distances[i])
    if not starts:
        scan_results[label] = {"energies": energies, "chem": None, "relaxed": None}
        print(f"{label:<10} unbound in this window" + ("" if dispersion_active else " (dispersion inactive)"))
        continue

    relaxed = relax_registry(film_at(label, starts.get("chem", starts.get("phys"))))
    occupied, buckling = carbon_sites_and_buckling(relaxed)
    if label in REGISTRY_SITES and occupied != REGISTRY_SITES[label]:
        scan_results[label] = {"energies": energies, "chem": starts.get("chem"), "relaxed": None}
        print(f"{label:<10} relaxed onto {occupied}: not a {label} result, dropped")
        continue
    scan_results[label] = {"energies": energies, "chem": starts.get("chem"), "relaxed": {
        "w_adh": calculate_adhesion_energy(relaxed, slab_relaxed, film_relaxed, calculator) * EV_PER_A2_TO_J_PER_M2,
        "separation": get_average_interlayer_distance(
            to_ase(relaxed), InterfacePartsEnum.SUBSTRATE.value, InterfacePartsEnum.FILM.value),
        "buckling": buckling,
        "material": relaxed,
    }}
    r = scan_results[label]["relaxed"]
    print(f"{label:<10} relaxed: d = {r['separation']:5.2f} A   buckling = {buckling_text(buckling)} A   "
          f"W_adh = {r['w_adh']:.2f} J/m^2")


In [ ]:
import plotly.graph_objects as go

reference = min(float(r["energies"].min()) for r in scan_results.values())
fig = go.Figure()
for label, r in scan_results.items():
    fig.add_trace(go.Scatter(x=distances, y=(r["energies"] - reference) * 1000 / n_carbon,
                             mode="lines+markers", name=label))
fig.update_layout(
    title="Rigid-scan energy vs. separation (bracketing only; the table below is relaxed)",
    xaxis_title="plane distance (A)",
    yaxis_title="energy above the deepest scanned point (meV / C atom)",
)
fig.show()


In [ ]:
# Lahiri et al. (2011), Table 1
PAPER = {"atop_fcc": (0.81, 2.16), "atop_hcp": (0.77, 2.17), "hollow": (0.31, 3.26)}
PAPER_BUCKLING_FCC = 0.03  # A, atop_fcc, computed (Lahiri et al. ref 35)

rows = {label: r["relaxed"] for label, r in scan_results.items() if r["relaxed"]}
print(f"{'registry':<10}{'W_adh':>7}{'paper':>7}     {'d':>5}{'paper':>7}    buckling")
for label, r in sorted(rows.items(), key=lambda kv: -kv[1]["w_adh"]):
    w, d = PAPER.get(label, ("—", "—"))
    print(f"{label:<10}{r['w_adh']:>7.2f}{w:>7}     {r['separation']:>5.2f}{d:>7}    {buckling_text(r['buckling'])}")
for label in set(scan_results) - set(rows):
    w, d = PAPER.get(label, ("—", "—"))
    print(f"{label:<10}no result here — paper: {w} J/m^2 at {d} A")

mace_reproduces = (
    all(label in rows for label in PAPER)
    and rows["atop_fcc"]["w_adh"] > rows["atop_hcp"]["w_adh"] > rows["hollow"]["w_adh"]
    and abs(rows["atop_fcc"]["w_adh"] - PAPER["atop_fcc"][0]) <= 0.15
    and abs(rows["atop_fcc"]["separation"] - PAPER["atop_fcc"][1]) <= 0.05
    and rows["atop_fcc"]["buckling"] is not None
    and 0.5 <= rows["atop_fcc"]["buckling"] / PAPER_BUCKLING_FCC <= 2
)


## 5. Precise Tier: the Paper's LDA, Relaxed, on the Platform

One **fixed-cell relaxation** per selected registry, starting from the MACE-relaxed geometry, at the
paper's functional — LDA, spin-polarized, no dispersion correction — plus the two same-cell
references the work of adhesion needs. Each job's final structure is read back, so separation and
buckling are compared as well as the energy. The graphene reference runs without spin polarization:
it is non-magnetic, and a symmetric spin-polarized solution has the same energy.

Divergences from Lahiri et al.: 4 Ni layers, not 5; 20 Å of vacuum, not 90; the platform relaxes
every atom, where the paper held the bottom two Ni layers; plane-wave ultrasoft pseudopotentials,
not all-electron LCAO.

A default run selects one registry (three jobs). An **empty** list skips the platform tier entirely,
which is what the automated test does: relaxations take longer than a browser test may wait.


In [ ]:
DFT_REGISTRY_NAMES = [
    "atop_fcc",
    # "atop_hcp",
    # "hollow",
    # "bridge",
]


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"Using project: {projects[0]['name']} ({project_id})")

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

def submitted_copy(material, name):
    """Drop the film/substrate labels: QE species names must match between input blocks."""
    m = material.clone()
    m.basis.labels.values = []
    m.name = name
    return Material.create(get_or_create_material(client, m, ACCOUNT_ID))

dft_materials, reference_materials = {}, {}
if DFT_REGISTRY_NAMES:
    for label in DFT_REGISTRY_NAMES:
        relaxed = scan_results[label]["relaxed"]
        if relaxed is None:
            print(f"{label:<16} skipped: no relaxed structure from the fast tier")
            continue
        saved = submitted_copy(relaxed["material"],
                               f"{BASE_MATERIAL_NAME} {label} d{relaxed['separation']:.2f} relaxed")
        dft_materials[label] = saved
        print(f"{label:<16} -> '{saved.name}' ({len(saved.basis.elements.values)} atoms)")
    for name, part in (("substrate", substrate_part), ("film", film_part)) if dft_materials else ():
        saved = submitted_copy(part, f"{BASE_MATERIAL_NAME} {name} reference")
        reference_materials[name] = saved
        print(f"{name + ' ref':<16} -> '{saved.name}' ({len(saved.basis.elements.values)} atoms)")
else:
    print("DFT tier skipped: no registries selected.")


In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

In [ ]:
from mat3ra.mode import ModelFactory
from mat3ra.standata.model_tree import ModelTreeStandata

# The paper's functional: LDA, no dispersion correction.
model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft",
    subtype=MODEL_SUBTYPE,
    functional=FUNCTIONAL,
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)


In [ ]:
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider

RELAX_UNIT = "pw_relax"

def configure(built, spin_polarized):
    """The published settings on the relaxation unit; a Ni moment only where there is Ni."""
    cutoffs = PlanewaveCutoffsContextProvider(wavefunction=ECUTWFC, density=ECUTRHO,
                                              isEdited=True).get_context_item_data()
    for subworkflow in built.subworkflows:
        subworkflow.model = model
        unit = subworkflow.get_unit_by_name(name=RELAX_UNIT)
        unit.add_context(cutoffs)
        subworkflow.set_unit(unit)
    apply_scf_kgrid(built, SCF_KGRID, material=reference_material, unit_name=RELAX_UNIT)
    system = {"degauss": DEGAUSS, "smearing": SMEARING, "nspin": 2 if spin_polarized else 1}
    if spin_polarized:
        system["starting_magnetization(1)"] = STARTING_MAGNETIZATION["Ni"]
    patch_workflow_qe_input(built, {"system": system, **ADDITIONAL_PARAMETERS}, unit_names=[RELAX_UNIT])
    return built

if dft_materials:
    reference_material = next(iter(dft_materials.values()))
    if reference_material.basis.elements.values[0] != "Ni":
        raise RuntimeError("Expected Ni as the first species — the magnetization index assumes it")
    configure(workflow, spin_polarized=True)


In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflows = {}
if dft_materials:
    film_workflow = Workflow.create(WorkflowStandata.filter_by_application(app.name)
                                    .get_by_name_first_match(WORKFLOW_SEARCH_TERM))
    film_workflow.name = f"{MY_WORKFLOW_NAME} film"
    workflows = {"interface": workflow, "substrate": workflow,
                 "film": configure(film_workflow, spin_polarized=False)}
    for key, built in workflows.items():
        saved_workflows[key] = Workflow.create(get_or_create_workflow(client, built, ACCOUNT_ID))
        print(f"{key:<12} -> workflow {saved_workflows[key].id}")


In [ ]:
if dft_materials:
    print(f"Available clusters: {[c['hostname'] for c in client.clusters.list()]}")


In [ ]:
from mat3ra.ide.compute import Compute

compute = None
if dft_materials:
    clusters = client.clusters.list()
    matching = [c for c in clusters if not CLUSTER_NAME or CLUSTER_NAME in c["hostname"]]
    if not matching:
        raise RuntimeError(f"No cluster matching {CLUSTER_NAME!r} is available; registered: "
                           f"{[c['hostname'] for c in clusters]}")
    compute = Compute(cluster=matching[0], queue=QUEUE_NAME, ppn=PPN)
    print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


In [ ]:
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.job import create_job

def submit_job_for(label, saved_material, which="interface"):
    job_response = create_job(
        api_client=client,
        materials=[saved_material],
        workflow=workflows[which],
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{MY_WORKFLOW_NAME} {label} {timestamp}",
        compute=compute.to_dict(),
    )
    job_id = dict_to_namespace_recursive(job_response)._id
    print(f"{label:<16} -> job {job_id}")
    return job_id

jobs, reference_jobs = {}, {}
if dft_materials:
    jobs = {label: submit_job_for(label, m) for label, m in dft_materials.items()}
    reference_jobs = {name: submit_job_for(f"{name} reference", m, which=name)
                      for name, m in reference_materials.items()}


In [ ]:
for label, job_id in {**jobs, **reference_jobs}.items():
    client.jobs.submit(job_id)
    print(f"Submitted {label}: {job_id}")


In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

all_job_ids = list(jobs.values()) + list(reference_jobs.values())
if all_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, all_job_ids, poll_interval=POLL_INTERVAL)
else:
    print("Nothing to wait for — the DFT tier was skipped.")


In [ ]:
from mat3ra.prode import PropertyName

RY_TO_EV = 13.605693123

def property_of(job_id, name):
    properties = client.properties.get_for_job(job_id, property_name=name)
    if not properties:
        raise RuntimeError(f"Job {job_id} reported no '{name}'")
    return properties[-1]

def total_energy_of(job_id):
    energy = property_of(job_id, PropertyName.scalar.total_energy.value)
    value, units = float(energy["value"]), str(energy.get("units", "eV")).lower()
    return value * RY_TO_EV if units.startswith("ry") else value

def final_structure_of(job_id):
    structure = property_of(job_id, PropertyName.non_scalar.final_structure.value)
    return Material.create(client.materials.get(structure["materialId"]))

def dft_geometry(material, label):
    """Separation, signed buckling (None without an atop carbon) and the sites the carbons occupy,
    read against the relaxed structure's own Ni — the submitted copy carries no labels."""
    cartesian = material.clone()
    cartesian.to_cartesian()
    pos = np.array(cartesian.basis.coordinates.values)
    elements = cartesian.basis.elements.values
    ni = [i for i, e in enumerate(elements) if e in substrate_elements]
    carbon = [i for i, e in enumerate(elements) if e in film_elements]
    top_ni = [i for i in ni if pos[i, 2] > max(pos[j, 2] for j in ni) - 0.5]
    separation = float(pos[carbon, 2].mean() - pos[top_ni, 2].mean())
    substrate = cartesian.clone()
    substrate.basis.filter_atoms_by_ids(ni)
    analyzer = SurfaceSiteAnalyzer(material=substrate)
    sites = {i: analyzer.get_site_name(pos[i, :2]) for i in carbon}
    atop = next((i for i, site in sites.items() if site == "atop"), None)
    buckling = None if atop is None else float(pos[atop, 2] - pos[next(i for i in carbon if i != atop), 2])
    return separation, buckling, set(sites.values())


In [ ]:
from mat3ra.made.tools.analyze.other import get_surface_area

area = get_surface_area(to_ase(base_interface))

dft_results = {}
if jobs:
    reference_energies = {name: total_energy_of(job_id) for name, job_id in reference_jobs.items()}
    separated = reference_energies["substrate"] + reference_energies["film"]
    for label, job_id in jobs.items():
        energy = total_energy_of(job_id)
        separation, buckling, occupied = dft_geometry(final_structure_of(job_id), label)
        drifted = label in REGISTRY_SITES and occupied != REGISTRY_SITES[label]
        if drifted:
            print(f"! {label}: carbons relaxed onto {occupied}, not {REGISTRY_SITES[label]} — excluded from the verdict")
        dft_results[label] = {"energy": energy, "w_adh": (separated - energy) / area * EV_PER_A2_TO_J_PER_M2,
                              "separation": separation, "buckling": buckling, "drifted": drifted}

    print(f"{'registry':<10}{'E (eV)':>12}{'W_adh':>8}{'paper':>7}     {'d':>5}{'paper':>7}    buckling")
    for label, r in sorted(dft_results.items(), key=lambda kv: -kv[1]["w_adh"]):
        w, d = PAPER.get(label, ("—", "—"))
        print(f"{label:<10}{r['energy']:>12.4f}{r['w_adh']:>8.2f}{w:>7}     {r['separation']:>5.2f}{d:>7}    {buckling_text(r['buckling'])}")


## 6. Compare with the Article


In [ ]:
# One verdict per tier against Lahiri et al. (2011) Table 1, reached through the review.
print("Targets: fcc 0.81 J/m^2 @ 2.16 A · hcp 0.77 @ 2.17 · hollow 0.31 @ 3.26 · "
      "buckling ~0.03 A, atop carbon out\n")
print(f"Reproduces Lahiri et al. Table 1 [MACE tier]: {'yes' if mace_reproduces else 'no'}")
print("(MACE is PBE-grade; the LDA tier carries the reproduction claim.)\n")

if dft_results:
    checks = []
    for label, r in dft_results.items():
        if label not in PAPER or r["drifted"]:
            continue
        w, d = PAPER[label]
        checks.append(abs(r["w_adh"] - w) <= 0.15 and abs(r["separation"] - d) <= 0.05)
        if label.startswith("atop"):
            checks.append(r["buckling"] is not None and 0.5 <= r["buckling"] / PAPER_BUCKLING_FCC <= 2)
    if all(label in dft_results and not dft_results[label]["drifted"] for label in PAPER):
        checks.append(dft_results["atop_fcc"]["w_adh"] > dft_results["atop_hcp"]["w_adh"] > dft_results["hollow"]["w_adh"])
        print("evaluated: all three registries, including their ordering")
    else:
        print(f"evaluated: {', '.join(label for label, r in dft_results.items() if label in PAPER and not r['drifted'])} "
              "(all three registries, undrifted, are needed for the ordering check)")
    print(f"Reproduces Lahiri et al. Table 1 [DFT tier]: {'yes' if checks and all(checks) else 'no'}")
else:
    print("DFT tier: not run — select registries in DFT_REGISTRY_NAMES for the paper's-functional verdict.")


## References

[1] Arjun Dahal, Matthias Batzill, "Graphene-nickel interfaces: a review",
Nanoscale 6(5), 2548 (2014). [DOI: 10.1039/c3nr05279f](https://doi.org/10.1039/c3nr05279f)

[2] Jayeeta Lahiri, Travis S. Miller, Andrew J. Ross, Lyudmyla Adamska, Ivan I. Oleynik,
Matthias Batzill, "Graphene growth and stability at nickel surfaces", New J. Phys. 13, 025001
(2011). [DOI: 10.1088/1367-2630/13/2/025001](https://doi.org/10.1088/1367-2630/13/2/025001)

[3] mat3ra-made: https://github.com/mat3ra/made

[4] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace
